# Макростатистика Росстата: локальный сбор и проверенные ряды

Парсер работает на локальном Mac. Восстановлены пять исходных HTML и каталог из
**260 Excel-ссылок**. Получены восемь основных файлов; архив услуг за 2022–2026 годы
обрабатывается с проверкой структуры каждого выпуска.

Разметка реальных листов сохранена в `sources/extraction_specs.json`.
Результат: **11 месячных и 3 квартальных показателя**. Месячные индексы ИПЦ с 1991 года,
цены производителей ОКВЭД2 с 2013 года, зарплата с 1998 года, текущий номинальный оборот
розницы, индексы физического объёма услуг; ВВП — в отдельной квартальной матрице.

HTTPS проверяется с официальной цепочкой Минцифры из `certificates/`.
Индекс розницы из `IFO_rozn_2025.xls` оказался годовым и исключён из месячной матрицы.
Целевой ряд ЦБ и его пресс-релизы здесь не загружаются.

## 1. Локальная среда

```bash
.venv/bin/jupyter lab notebooks/rosstat_macro.ipynb
```

В новом checkout сначала запустите `bash run_local.sh`, затем установите полный
набор зависимостей: `.venv/bin/python -m pip install -r requirements.txt`.
Для повторной обработки загруженных файлов установите `ALLOW_NETWORK=False` ниже.
Системное хранилище сертификатов не изменяется.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import sys
import warnings
warnings.filterwarnings("once", message="The xlrd engine is deprecated.*")
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'rosstat_parser.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'rosstat_parser.py').exists():
    raise RuntimeError('Запустите Jupyter из корня репозитория или папки notebooks')
sys.path.insert(0, str(ROOT))
from rosstat_parser import (
    catalog_from_saved_html, download_selected, parse_source, combine, suggest_layouts,
)
DATA = ROOT / 'data'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RAW, OUT = DATA / 'raw', DATA / 'processed' / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_colwidth', 160)
print('Результаты этого запуска:', OUT)

Результаты этого запуска: /Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/data/processed/20261029T205132845965Z


## 2. Разбор сохранённых HTML и применение проверенного отбора
Локальные копии вложений лежат в `data/input/` и не входят в Git.
Если их нет в новом checkout, используется переносимый каталог `sources/macro_catalog.csv`.
HTML читается как данные: его JavaScript и инструкции не исполняются.

Из карточек извлекаются название, ссылка, дата карточки, раздел и год группы.
Например, название «Сентябрь» получает контекст «Платные услуги / 2026 год».
Дата карточки **не является датой выхода каждого исторического наблюдения**.
`frequency_hint` — подсказка из текста; `target_frequency` — требуемая частота;
ни одно из этих полей не подтверждает структуру самого Excel.

Отбор фиксируется по точным URL и заголовкам. Новые или изменённые документы
не выбираются автоматически. Скрипт `prepare_sources.py` воспроизводит сохранённый
каталог из исходных вложений и проверяет их SHA-256.

In [2]:
registry = pd.read_csv(ROOT / 'sources' / 'macro_catalog.csv').fillna('')
page_specs = json.loads((ROOT / 'sources' / 'macro_pages.json').read_text())
frames = []
for page in page_specs:
    path = DATA / 'input' / page['input_name']
    if path.exists():
        current = catalog_from_saved_html(path, page['page_url'], page['topic'])
        actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        if actual_hash != page['input_sha256']:
            raise RuntimeError(f"Изменилось вложение {path.name}: сначала пересмотрите отбор источников")
    else:
        current = registry.loc[registry.topic.eq(page['topic'])].copy()
    frames.append(current)
catalog = pd.concat(frames, ignore_index=True).drop_duplicates('url').reset_index(drop=True)
review = registry.set_index('url')
for column in ['role', 'indicator_hint', 'target_frequency', 'selection_reason']:
    catalog[column] = catalog.url.map(review[column]).fillna('not_reviewed' if column == 'role' else '')
catalog['file_verified'] = False  # Статус описывает каталог, а не результаты последующей загрузки.
catalog.to_csv(OUT / 'source_catalog.csv', index=False)
print('Уникальных Excel-ссылок:', len(catalog))
display(catalog.groupby(['topic', 'role']).size().rename('files').reset_index())

Уникальных Excel-ссылок: 260


,topic,role,files
0,accounts,not_selected,12
1,accounts,optional,2
2,accounts,primary,1
3,labour,not_selected,13
4,labour,primary,1
5,price,not_selected,63
6,price,optional,2
7,price,primary,3
8,retail,not_selected,39
9,retail,optional,2


## 3. Отбор источников

Восемь файлов `primary` закрывают первый просмотр всех пяти тем.
`INCLUDE_SERVICES_HISTORY=True` добавляет 56 архивных выпусков платных услуг.
Семь `optional` источников остаются в каталоге для последующего расширения.

Проверка реальных файлов показала:
- `IFO_rozn_2025.xls` содержит годовые, а не месячные индексы;
- текущий оборот розницы содержит только месяцы 2026 года и выражен в текущих ценах;
- зарплата до 1998 года выражена в тысячах рублей, поэтому рублёвый ряд начинается с 1998;
- старые классификации ОКВЭД и разные базовые цены ВВП не склеиваются;
- в каждом выпуске услуг берутся два индекса за отдельный месяц; накопительный индекс исключён.

Промпроизводство, безработица, базовый ИПЦ, реальные доходы и инвестиции требуют расширения
набора источников. История разных показателей имеет разную длину; пропуски не заполняются.

In [3]:
INCLUDE_SERVICES_HISTORY = True
ROLES = ['primary', 'services_history'] if INCLUDE_SERVICES_HISTORY else ['primary']
selected = catalog.loc[catalog.role.isin(ROLES)].copy()
if selected.empty:
    raise RuntimeError('Отбор пуст')
display(selected[['topic', 'display_title', 'file_name', 'target_frequency', 'url']])
selected.to_csv(OUT / 'selected_sources.csv', index=False)
print('Файлов для получения:', len(selected))

,topic,display_title,file_name,target_frequency,url
1,accounts,ВВП кварталы (с 1995 г.),VVP_kvartal_s-1995-2026.xlsx,quarterly,https://rosstat.gov.ru/storage/mediabank/VVP_kvartal_s-1995-2026.xlsx
16,price,"Индексы потребительских цен на товары и услуги по Российской Федерации, федеральным округам и субъектам Российской Федерации (с 2022 г.)",ipc_RF_fo_sub_09-2026.xlsx,monthly,https://rosstat.gov.ru/storage/mediabank/ipc_RF_fo_sub_09-2026.xlsx
17,price,"Индексы потребительских цен на товары и услуги по Российской Федерации, месяцы (с 1991 г.)",ipc_mes_09-2026.xlsx,monthly,https://rosstat.gov.ru/storage/mediabank/ipc_mes_09-2026.xlsx
59,price,Индексы цен производителей по видам экономической деятельности с 1998 г.,Proizvoditeli_Ind_VED_09-2026.xlsx,monthly,https://rosstat.gov.ru/storage/mediabank/Proizvoditeli_Ind_VED_09-2026.xlsx
83,labour,Среднемесячная номинальная начисленная заработная плата работников в целом по экономике Российской Федерации в 1991-2026 гг.,tab1-zpl_08-2026.xlsx,monthly,https://rosstat.gov.ru/storage/mediabank/tab1-zpl_08-2026.xlsx
...,...,...,...,...,...
194,services,"Платные услуги населению / Основные показатели платных услуг населению по видам в разрезе субъектов Российской Федерации, месяцы (с 2022 г.) / 2022 год / Май",plat_052022.xls,monthly,https://rosstat.gov.ru/storage/mediabank/plat_052022.xls
195,services,"Платные услуги населению / Основные показатели платных услуг населению по видам в разрезе субъектов Российской Федерации, месяцы (с 2022 г.) / 2022 год / Ап...",plat_042022.xls,monthly,https://rosstat.gov.ru/storage/mediabank/plat_042022.xls
196,services,"Платные услуги населению / Основные показатели платных услуг населению по видам в разрезе субъектов Российской Федерации, месяцы (с 2022 г.) / 2022 год / Март",plat_032022.xls,monthly,https://rosstat.gov.ru/storage/mediabank/plat_032022.xls
197,services,"Платные услуги населению / Основные показатели платных услуг населению по видам в разрезе субъектов Российской Федерации, месяцы (с 2022 г.) / 2022 год / Фе...",plat_022022.xls,monthly,https://rosstat.gov.ru/storage/mediabank/plat_022022.xls


Файлов для получения: 64


## 4. Получение Excel: локальные файлы, кеш или HTTPS
Положите скачанные файлы в `data/input/excel/`, сохранив **точные исходные имена**.
Загрузчик сначала импортирует локальные файлы, затем проверяет кеш, затем обращается
к сайту. При импорте происхождение помечается `user_supplied_file`: соответствие URL
установлено по имени, а не подтверждено скачиванием через HTTPS.

Сохраняются исходные байты, SHA-256 и квитанция получения. При общей ошибке сертификата
прокси выполняется одна неудачная сетевая попытка; остальные сетевые загрузки помечаются
как **не выполненные**, но локальные файлы продолжают импортироваться.
`ALLOW_NETWORK=False` позволяет работать полностью офлайн. `REFRESH=True` обновляет
сетевой снимок; старые файлы с другими хешами сохраняются.

Тот же загрузчик доступен без Jupyter:
```bash
.venv/bin/python download_sources.py
.venv/bin/python download_sources.py --offline --local-dir data/input/excel
```
Код завершения CLI — 2, если получены не все выбранные файлы, включая пропущенные загрузки.

In [4]:
ALLOW_NETWORK = False
REFRESH = False
LOCAL_EXCEL = DATA / 'input' / 'excel'
downloads, download_report = download_selected(
    selected, RAW, local_dir=LOCAL_EXCEL, refresh=REFRESH, allow_network=ALLOW_NETWORK,
)
download_report.to_json(OUT / 'download_report.json', orient='records', force_ascii=False, indent=2)
display(download_report)
print(f'Доступно файлов: {len(downloads)} / {selected.url.nunique()}')
if not downloads:
    print('СТАТИСТИКА НЕ ПОЛУЧЕНА. Следующие ячейки не создадут выдуманную или пустую выгрузку.')
if len(downloads) != selected.url.nunique():
    raise RuntimeError('Получены не все выбранные файлы; см. download_report')

,source_id,file_name,url,status,sha256
0,3ac820c599b53681,VVP_kvartal_s-1995-2026.xlsx,https://rosstat.gov.ru/storage/mediabank/VVP_kvartal_s-1995-2026.xlsx,cached,66eaab3f7b687fa09b427a020d038698e85665a669bf1e64bb6fa37a006086b0
1,e70fa71d16ba71ef,ipc_RF_fo_sub_09-2026.xlsx,https://rosstat.gov.ru/storage/mediabank/ipc_RF_fo_sub_09-2026.xlsx,cached,ef271e18e7ea72e3ed0ff89bf743c61ef5f254e8757ea10bc0a9bb2d58fb7a82
2,3f55908017a5c59d,ipc_mes_09-2026.xlsx,https://rosstat.gov.ru/storage/mediabank/ipc_mes_09-2026.xlsx,cached,540553b4fce35ed2c57e5497f1f1fa1011c3da4dcf39d7191185760568f97d3e
3,61b24a0a7dd7a252,Proizvoditeli_Ind_VED_09-2026.xlsx,https://rosstat.gov.ru/storage/mediabank/Proizvoditeli_Ind_VED_09-2026.xlsx,cached,e57c5ac2942d603fca1a620c715dea51ddcc957f955032178537978d2ddbcf33
4,59876b4c05b6901e,tab1-zpl_08-2026.xlsx,https://rosstat.gov.ru/storage/mediabank/tab1-zpl_08-2026.xlsx,cached,27e2d33eae4439b89805a08b45e7bd465b8ca45257fc09575a52e07c9504fd33
...,...,...,...,...,...
59,4a5544993bf09007,plat_052022.xls,https://rosstat.gov.ru/storage/mediabank/plat_052022.xls,cached,5d583b34b49213ea3a956ae8371a80295d7e4eb1c0f366efccd9b71f6d34136b
60,66b63243e55c5e5b,plat_042022.xls,https://rosstat.gov.ru/storage/mediabank/plat_042022.xls,cached,c268cac82dde0c288c857f62850fa0caa986ec85192db04ae6a195a78646a559
61,a3279fb40d3a9485,plat_032022.xls,https://rosstat.gov.ru/storage/mediabank/plat_032022.xls,cached,bee036c89d3b52a3cd4f3cf885220d09516fa571dbfd425ed442f3736ca0f134
62,1527ea0b6ef922f5,plat_022022.xls,https://rosstat.gov.ru/storage/mediabank/plat_022022.xls,cached,765669f199e859aa5eec6b670d2e20cbc9b20e30ccb2c870e20e7de47824f1f5


Доступно файлов: 64 / 64


## 5. Просмотр листов и разметка
Для каждого файла проверьте географию, частоту, единицу, базу индекса и методологию.
Предпросмотр ограничен первыми тремя листами; полный список имён выводится отдельно.
Индексы строк/столбцов начинаются с **нуля**.
После выбора листа можно открыть его целиком и получить структурные подсказки:
```python
path, metadata = downloads[нужный_url]
frame = pd.read_excel(path, sheet_name='Фактическое имя', header=None)
display(frame.head(40))
suggest_layouts(frame)  # Подсказки месячных раскладок, не выбор экономического ряда.
```
Для квартального ВВП задайте `frequency='quarterly'`. Квартальные значения хранятся
отдельно и не интерполируются в месяцы. Накопительные полугодовые/девятимесячные итоги
не являются отдельными кварталами.

In [5]:
# Показываем основные книги; архив услуг проверяется при извлечении.
for url in selected.loc[selected.role.eq('primary'), 'url']:
    path, metadata = downloads[url]
    with pd.ExcelFile(path) as book:
        print(url.rsplit('/', 1)[-1], '— листов:', len(book.sheet_names), book.sheet_names[:5])
        sheet = next((s for s in book.sheet_names if s != 'Содержание'), book.sheet_names[0])
        display(pd.read_excel(book, sheet_name=sheet, header=None, nrows=9).iloc[:, :10])

VVP_kvartal_s-1995-2026.xlsx — листов: 16 ['Содержание', '1', '2', '3', '4']


,0,1,2,3,4,5,6,7,8,9
0,К содержанию,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Валовой внутрений продукт1)\n(в текущих ценах, млрд.руб., до 1998г. -трлн.руб.)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1995,NaN,NaN,NaN,1996,NaN,NaN,NaN,1997,NaN
3,I квартал,II квартал,III квартал,IV квартал,I квартал,II квартал,III квартал,IV квартал,I квартал,II квартал
4,234.9775,324.252184,421.059675,448.232707,425.2889,468.4185,548.8781,565.2396,512.3688,555.0768
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1) Данные за кварталы 1995-2002гг. не пересматривались и могут не соответствовать данным за кварталы 2003-2011 гг.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


ipc_RF_fo_sub_09-2026.xlsx — листов: 58 ['Содержание', '01(2022)', '02(2022)', '03(2022)', '04(2022)']


,0,1,2,3,4,5
0,К содержанию,NaN,"Индексы потребительских цен на товары и услуги \nпо Российской Федерации, федеральным округам \nи субъектам Российской Федерации в январе 2022 г.",NaN,NaN,NaN
1,NaN,NaN,"на конец периода, в % к предыдущему месяцу",NaN,NaN,NaN
2,Код территории,Наименование территории,ИПЦ,в том числе:,NaN,NaN
3,NaN,NaN,NaN,продовольственные товары,непродовольственные товары,услуги
4,643,Российская Федерация,100.99,101.44,100.67,100.76
5,30,Центральный федеральный округ,100.96,101.26,100.74,100.83
6,14000000,Белгородская область,101.03,101.27,100.96,100.71
7,15000000,Брянская область,101.46,101.27,101.15,102.24
8,17000000,Владимирская область,101.08,101.49,100.94,100.5


ipc_mes_09-2026.xlsx — листов: 5 ['Содержание', '01', '02', '03', '04']


...

/var/folders/zn/0ybb4xqd5wx4_l54mrhtxwhr0000gn/T/ipykernel_29750/1545405418.py:4: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  with pd.ExcelFile(path) as book:


,0,1,2,3,4,5
0,К содержанию,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,"Индекс физического объема оборота розничной торговли по Российской Федерации, в % к предыдущему году (в сопоставимых ценах)",NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Оборот розничной торговли,NaN,NaN,NaN,NaN
5,NaN,всего,в том числе,NaN,Оборот розничной торговли торгующих организаций³⁾,продажа товаров на розничных рынках и ярмарках
6,NaN,NaN,"Пищевыми продуктами, включая напитки, и табачные изделия¹⁾",Непродовольственными товарами²⁾,NaN,NaN
7,1971,107,105,109.5,106.5,128.1
8,1972,106.7,105.8,107.9,106.8,103.8


plat_092026.xls — листов: 23 ['Содержание', 'платные всего', 'бытовые всего', 'транспортные', 'почтовые']


/var/folders/zn/0ybb4xqd5wx4_l54mrhtxwhr0000gn/T/ipykernel_29750/1545405418.py:4: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  with pd.ExcelFile(path) as book:


,0,1,2,3,4
0,К содержанию,NaN,NaN,NaN,NaN
1,Основные показатели платных услуг населению за сентябрь 2026 г.,NaN,NaN,NaN,NaN
2,Платные услуги населению,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,"Фактически, тысяч рублей",Индекс физического объема \n% к (в сопоставимых ценах):,NaN,NaN
5,NaN,NaN,предыдущему месяцу,соответствующему месяцу прошлого года,соответствующему периоду прошлого года
6,Россия,1891143294.7,99.5,101.9,102.3
7,Центральный федеральный округ,716488427.3,100.2,103.8,102.5
8,Белгородская область,12349800.7,100.9,101.7,103.4


## 6. Извлечение проверенных рядов

`extraction_specs.json` задаёт лист, единицу, базу, территорию, координаты и контрольные
заголовки для реальных таблиц. Для каждого архивного выпуска услуг дополнительно
проверяются месяц/год, строка России и база индекса.

Пустые периоды в конце текущего года отсекаются; внутренние пропуски сохраняются.
ВВП не интерполируется в месяцы. Неисправная разметка или изменение структуры дают ошибку.

In [6]:
from extract_dataset import extraction_specs, export_dataset
SOURCE_SPECS = extraction_specs(ROOT, selected, downloads, INCLUDE_SERVICES_HISTORY)
print('Размеченных источников/листов:', len(SOURCE_SPECS))
print('Показателей:', len({s['indicator'] for s in SOURCE_SPECS}))

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is de

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)


/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)


/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is de

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)


...

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is de

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)


/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)


Размеченных источников/листов: 129
Показателей: 14


/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name='платные всего', header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:28: Pandas4Warning: The xlrd engine is de

## 7. Проверка качества и экспорт

`macro_long.csv` сохраняет происхождение каждого значения: URL, SHA-256, лист,
время получения, частоту, единицу и базу. `macro_monthly.csv` и `macro_quarterly.csv`
выгружаются отдельно. `quality_report.csv` показывает фактический диапазон и пропуски.

Дополнительная проверка сравнивает четыре длинных ряда ИПЦ с независимыми строками
России из региональной книги. При несовпадении экспорт останавливается.
`source_assessment.json` сохраняет методологические и территориальные примечания.
Каждый запуск получает отдельную папку.

In [7]:
macro, quality = export_dataset(ROOT, selected, downloads, SOURCE_SPECS, OUT)
display(quality)
display(macro[['indicator', 'period', 'value', 'frequency', 'unit']].tail(12))
print('Числовых наблюдений:', int(macro.value.notna().sum()))
print('Проверка ИПЦ:', json.loads((OUT / 'validation.json').read_text()))
print('Результаты:', OUT)

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)


/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecat

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecat

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecat

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(path, sheet_name=spec['sheet'], header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/rosstat_parser.py:286: Pandas4Warning: The xlrd engine is deprecat

...

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd

/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)
/Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/extract_dataset.py:120: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  frame = pd.read_excel(downloads[url][0], sheet_name=sheet, header=None)


,indicator,frequency,start,end,observed,missing
0,cpi_food_mom_index,monthly,1991-01-01,2026-09-01,429,0
1,cpi_mom_index,monthly,1991-01-01,2026-09-01,429,0
2,cpi_nonfood_mom_index,monthly,1991-01-01,2026-09-01,429,0
3,cpi_services_mom_index,monthly,1991-01-01,2026-09-01,429,0
4,gdp_real_2021_bln_rub,quarterly,2011-01-01,2026-04-01,62,0
5,gdp_real_2021_sa_bln_rub,quarterly,2011-01-01,2026-04-01,62,0
6,gdp_volume_yoy_index,quarterly,2012-01-01,2026-04-01,58,0
7,nominal_wages_rub,monthly,1998-01-01,2026-08-01,344,0
8,paid_services_volume_mom_index,monthly,2022-01-01,2026-09-01,57,0
9,paid_services_volume_yoy_index,monthly,2022-01-01,2026-09-01,57,0


,indicator,period,value,frequency,unit
2848,ppi_mining_mom_index,2026-07-01,85.83,monthly,%
2849,ppi_mining_mom_index,2026-08-01,101.94,monthly,%
2850,ppi_mining_mom_index,2026-09-01,110.50,monthly,%
2851,retail_turnover_nominal_mln_rub,2026-01-01,4912104.30,monthly,млн рублей
2852,retail_turnover_nominal_mln_rub,2026-02-01,4844375.20,monthly,млн рублей
2853,retail_turnover_nominal_mln_rub,2026-03-01,5591362.50,monthly,млн рублей
2854,retail_turnover_nominal_mln_rub,2026-04-01,5476231.90,monthly,млн рублей
2855,retail_turnover_nominal_mln_rub,2026-05-01,5700805.00,monthly,млн рублей
2856,retail_turnover_nominal_mln_rub,2026-06-01,5773141.30,monthly,млн рублей
2857,retail_turnover_nominal_mln_rub,2026-07-01,6037422.30,monthly,млн рублей


Числовых наблюдений: 2860
Проверка ИПЦ: {'independent_cpi_values_checked': 228, 'matched': True}
Результаты: /Users/oleynik_n_a/Documents/ChatGPT/Годовой проект/data/processed/20261029T205132845965Z


## 8. Использование в прогнозе
`period` — период наблюдения, `retrieved_at` — время получения снимка.
Ни одно из этих полей не заменяет фактическую дату и время публикации.
`release_date` оставлена пустой; дата карточки и HTTP Last-Modified её не восстанавливают.

Для ретроспективного теста нужны календарь публикаций и архив версий.
Современные исторические ряды могут содержать пересмотры. Не объединяйте их с дневным
курсом по месяцу наблюдения. `merge_asof` допустим только после установления момента
доступности данных и его часового пояса. Лаг на один месяц не гарантирует отсутствия утечки.

Проверки парсера выполняются командой `python -m pytest -q`.
Они используют искусственные таблицы и не подтверждают корректность реальных листов Росстата.
Следующий этап — расширение набора источников и восстановление дат публикаций для ретроспективного теста.